In [ ]:
import gurobipy as gp
from gurobipy import GRB

items = ['A', 'B']
months = [3, 4, 5, 6]

demand = {
    'A': {3: 400, 4: 500, 5: 600, 6: 400},
    'B': {3: 600, 4: 600, 5: 700, 6: 600}
}
holding_cost = {'A': 1.20, 'B': 1.00}
initial_inventory = {'A': 100, 'B': 150}
storage_capacity = 250
min_ending_inventory_B = 150

# --- [수정] 월별로 다른 생산 능력 및 생산 비용 파라미터 ---
prod_capacity = {
    'A': 550,
    'B': {3: 650, 4: 650, 5: 700, 6: 700}
}
mfg_cost_B = {
    3: 8.00, 4: 8.00,
    5: 6.50, 6: 6.50
}

model = gp.Model("Production_Planning_New_System")

p = model.addVars(items, months, name="produce")
i = model.addVars(items, months, name="inventory")

# --- [수정] 총비용(재고비 + B제품 생산비)을 최소화하는 목적 함수 ---
inventory_cost = gp.quicksum(holding_cost[j] * i[j, t] for j in items for t in months)
production_cost_B = gp.quicksum(mfg_cost_B[t] * p['B', t] for t in months)
model.setObjective(inventory_cost + production_cost_B, GRB.MINIMIZE)

for j in items:
    for t in months:
        if t == 3:
            model.addConstr(
                initial_inventory[j] + p[j, t] - demand[j][t] == i[j, t],
                name=f"InvBalance_{j}_{t}"
            )
        else:
            model.addConstr(
                i[j, t - 1] + p[j, t] - demand[j][t] == i[j, t],
                name=f"InvBalance_{j}_{t}"
            )

# --- [수정] 월별로 다른 B제품 생산 능력을 적용하는 제약식 ---
model.addConstrs((p['A', t] <= prod_capacity['A'] for t in months), name="ProdCap_A")
model.addConstrs((p['B', t] <= prod_capacity['B'][t] for t in months), name="ProdCap_B")

for t in months:
    model.addConstr(
        gp.quicksum(i[j, t] for j in items) <= storage_capacity,
        name=f"StorageCap_{t}"
    )

model.addConstr(i['B', 6] >= min_ending_inventory_B, name="EndingInv_B")

model.optimize()

if model.Status == GRB.OPTIMAL:
    print("\n-------------------------------------------")
    # --- [수정] 출력 메시지 변경 ---
    print(f"Total Cost (Inventory + Production): ${model.ObjVal:,.2f}")
    print("-------------------------------------------")
    print("Optimal Solution:")
    print(f"{'Month':<5} {'Item':<5} {'Production':>10} {'Inventory':>12}")
    print("-" * 40)
    for t in months:
        for j in items:
            prod_val = p[j, t].X if p[j, t].X > 1e-6 else 0
            inv_val = i[j, t].X if i[j, t].X > 1e-6 else 0
            print(f"{t}   {j:<5} {prod_val:>10.0f} {inv_val:>12.0f}")

In [3]:
import gurobipy as gp
from gurobipy import GRB

# --- 공통 파라미터 정의 ---
items = ['A', 'B']
months = [3, 4, 5, 6]
demand = {
    'A': {3: 400, 4: 500, 5: 600, 6: 400},
    'B': {3: 600, 4: 600, 5: 700, 6: 600}
}
holding_cost = {'A': 1.20, 'B': 1.00}
initial_inventory = {'A': 100, 'B': 150}
storage_capacity = 250
min_ending_inventory_B = 150

# --- 시나리오 1: 기존 시스템 (Original System) ---
print("--- 1. 기존 시스템 총비용 계산 중... ---")

model_orig = gp.Model("Original_System")
model_orig.setParam('OutputFlag', 0) # Gurobi 로그 출력 끄기

p_orig = model_orig.addVars(items, months, name="produce")
i_orig = model_orig.addVars(items, months, name="inventory")

# 기존 시스템의 B제품 생산 비용 ($8.00 고정)
mfg_cost_B_orig = 8.00

inventory_cost_orig = gp.quicksum(holding_cost[j] * i_orig[j, t] for j in items for t in months)
production_cost_B_orig = gp.quicksum(mfg_cost_B_orig * p_orig['B', t] for t in months)
model_orig.setObjective(inventory_cost_orig + production_cost_B_orig, GRB.MINIMIZE)

# 기존 시스템의 생산 능력 (A: 550, B: 650 고정)
prod_capacity_orig = {'A': 550, 'B': 650}

# 제약식 (기존 시스템)
for j in items:
    for t in months:
        if t == 3:
            model_orig.addConstr(initial_inventory[j] + p_orig[j, t] - demand[j][t] == i_orig[j, t])
        else:
            model_orig.addConstr(i_orig[j, t - 1] + p_orig[j, t] - demand[j][t] == i_orig[j, t])
    model_orig.addConstrs((p_orig[j, t] <= prod_capacity_orig[j] for t in months), name=f"ProdCap_{j}")

model_orig.addConstrs((gp.quicksum(i_orig[j, t] for j in items) <= storage_capacity for t in months))
model_orig.addConstr(i_orig['B', 6] >= min_ending_inventory_B)

model_orig.optimize()
total_cost_original = model_orig.ObjVal

# --- 시나리오 2: 새로운 시스템 (New System) ---
print("--- 2. 새로운 시스템 총비용 계산 중... ---")

model_new = gp.Model("New_System")
model_new.setParam('OutputFlag', 0)

p_new = model_new.addVars(items, months, name="produce")
i_new = model_new.addVars(items, months, name="inventory")

mfg_cost_B_new = {3: 8.00, 4: 8.00, 5: 6.50, 6: 6.50}
prod_capacity_new = {'A': 550, 'B': {3: 650, 4: 650, 5: 700, 6: 700}}

inventory_cost_new = gp.quicksum(holding_cost[j] * i_new[j, t] for j in items for t in months)
production_cost_B_new = gp.quicksum(mfg_cost_B_new[t] * p_new['B', t] for t in months)
model_new.setObjective(inventory_cost_new + production_cost_B_new, GRB.MINIMIZE)

for j in items:
    for t in months:
        if t == 3:
            model_new.addConstr(initial_inventory[j] + p_new[j, t] - demand[j][t] == i_new[j, t])
        else:
            model_new.addConstr(i_new[j, t - 1] + p_new[j, t] - demand[j][t] == i_new[j, t])

model_new.addConstrs((p_new['A', t] <= prod_capacity_new['A'] for t in months))
model_new.addConstrs((p_new['B', t] <= prod_capacity_new['B'][t] for t in months))
model_new.addConstrs((gp.quicksum(i_new[j, t] for j in items) <= storage_capacity for t in months))
model_new.addConstr(i_new['B', 6] >= min_ending_inventory_B)

model_new.optimize()
total_cost_new = model_new.ObjVal

# --- 3. 최종 결과 비교 ---
print("\n============================================")
print("     System Assessment")
print("============================================")
print(f"  - (a) cost: ${total_cost_original:12,.2f}")
print(f"  - (c) cost: ${total_cost_new:12,.2f}")
print("--------------------------------------------")
savings = total_cost_original - total_cost_new
print(f"  => total cost reduction:    ${savings:12,.2f}")
print("============================================")

--- 1. 기존 시스템 총비용 계산 중... ---
--- 2. 새로운 시스템 총비용 계산 중... ---

     System Assessment
  - (a) cost: $   20,560.00
  - (c) cost: $   18,210.00
--------------------------------------------
  => total cost reduction:    $    2,350.00


In [2]:
import gurobipy as gp
from gurobipy import GRB

try:
    # 1. 모델 객체 생성
    m = gp.Model("lp_problem")

    # 2. 변수 생성 (x1부터 x7까지)
    # addVars는 여러 변수를 한 번에 생성합니다. lb=0은 non-negativity 제약조건입니다.
    x = m.addVars(7, lb=0, name="x")

    # 3. 목적 함수 설정
    # setObjective는 목적 함수를 정의합니다. GRB.MINIMIZE는 최소화 문제입니다.
    m.setObjective(
        -0.75 * x[3] + 20 * x[4] - 0.5 * x[5] + 6 * x[6], 
        GRB.MINIMIZE
    )

    # 4. 제약 조건 추가
    # addConstr는 제약식을 추가합니다.
    m.addConstr(x[0] + 0.25 * x[3] - 8 * x[4] - x[5] + 9 * x[6] == 0, "c0")
    m.addConstr(x[1] + 0.5 * x[3] - 12 * x[4] - 0.5 * x[5] + 3 * x[6] == 0, "c1")
    m.addConstr(x[2] + x[5] == 1, "c2")
    
    # 참고: 변수 인덱스
    # x[0] -> x1, x[1] -> x2, x[2] -> x3
    # x[3] -> x4, x[4] -> x5, x[5] -> x6, x[6] -> x7

    # 5. 모델 최적화
    m.optimize()

    # 6. 결과 출력
    print("\n--- 최적화 결과 ---")
    # 변수별 최적해 출력
    # for v in m.getVars():
    #     # 변수 이름에서 인덱스를 추출하여 1부터 시작하도록 조정
    #     var_index = int(v.VarName[2:]) + 1
    #     print(f"x{var_index} = {v.X:.4f}")
    
    # 최적 목적 함수 값 출력
    print(f"\n최적 목적 함수 값 (Optimal Objective Value): {m.ObjVal:.4f}")

except gp.GurobiError as e:
    print(f"Error code {e.errno}: {e}")

except AttributeError:
    print("최적해를 찾지 못했습니다. (Infeasible or Unbounded)")

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[arm] - Darwin 25.0.0 25A362)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 3 rows, 7 columns and 12 nonzeros
Model fingerprint: 0x02001359
Coefficient statistics:
  Matrix range     [2e-01, 1e+01]
  Objective range  [5e-01, 2e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 1 rows and 5 columns
Presolve time: 0.00s
Presolved: 2 rows, 2 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -6.0000000e+30   2.000000e+30   6.000000e+00      0s
       3   -1.2500000e+00   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.00 seconds (0.00 work units)
Optimal objective -1.250000000e+00

--- 최적화 결과 ---

최적 목적 함수 값 (Optimal Objective Value): -1.2500


In [1]:
import gurobipy as gp
from gurobipy import GRB

# --- 1. 모델 생성 ---
model = gp.Model("LP_5.1")

# --- 2. 변수 정의 ---
x = model.addVars(5, lb=0, name="x")  # x[0]~x[4]은 x1~x5

# --- 3. 목적함수 (Minimize) ---
# Minimize: x1 + 4x2 - 7x3 + x4 + 5x5
model.setObjective(
    x[0] + 4*x[1] - 7*x[2] + x[3] + 5*x[4],
    GRB.MINIMIZE
)

# --- 4. 제약식 ---
# 1st: x1 - (3/4)x2 + 2x3 - (1/4)x4 = 6
model.addConstr(x[0] - 0.75*x[1] + 2*x[2] - 0.25*x[3] == 6, "c1")

# 2nd: - (1/4)x2 + 3x3 - (3/4)x4 + x5 = 5
model.addConstr(-0.25*x[1] + 3*x[2] - 0.75*x[3] + x[4] == 5, "c2")

# --- 5. 최적화 ---
model.optimize()

# --- 6. 결과 출력 ---
if model.status == GRB.OPTIMAL:
    print("\nOptimal Solution Found:")
    for i in range(5):
        print(f"x{i+1} = {x[i].X:.4f}")
    print(f"Objective Value (Z) = {model.ObjVal:.4f}")
else:
    print("No optimal solution found.")

Set parameter Username
Set parameter LicenseID to value 2611964
Academic license - for non-commercial use only - expires 2026-01-20
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[arm] - Darwin 25.0.0 25A362)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2 rows, 5 columns and 8 nonzeros
Model fingerprint: 0xecaa092f
Coefficient statistics:
  Matrix range     [2e-01, 3e+00]
  Objective range  [1e+00, 7e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+00, 6e+00]
Presolve removed 0 rows and 2 columns
Presolve time: 0.01s
Presolved: 2 rows, 3 columns, 6 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -7.0000000e+30   1.750000e+30   7.000000e+00      0s
       2   -1.9666667e+01   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.02 seconds (0.00 work units)
Optimal objective -1.966666667e+01

Optimal Solution Found:
x1 = 0.0000
x2 = 0.0000
x3 =

In [3]:
import gurobipy as gp
from gurobipy import GRB

# --- 1. 모델 생성 ---
model = gp.Model("LP_5_14")

# --- 2. 변수 정의 (bounded variables 포함) ---
# 각 변수의 하한(lbound), 상한(ubound)을 그대로 반영
x1 = model.addVar(lb=0, ub=4, name="x1")
x2 = model.addVar(lb=-2, ub=3, name="x2")
x3 = model.addVar(lb=2, ub=GRB.INFINITY, name="x3")

# --- 3. 목적함수 (Maximize) ---
model.setObjective(2*x1 + 3*x2 - 2*x3, GRB.MAXIMIZE)

# --- 4. 제약식 ---
model.addConstr(x1 + 3*x2 + x3 <= 8, "c1")
model.addConstr(2*x1 + x2 - x3 >= 3, "c2")

# --- 5. 최적화 ---
model.optimize()

# --- 6. 결과 출력 ---
if model.status == GRB.OPTIMAL:
    print("\nOptimal Solution:")
    for v in model.getVars():
        print(f"{v.varName} = {v.x:.4f}")
    print(f"Objective Value = {model.ObjVal:.4f}")
else:
    print("No optimal solution found.")

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[arm] - Darwin 25.0.0 25A362)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2 rows, 3 columns and 6 nonzeros
Model fingerprint: 0x86ac30e1
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [2e+00, 3e+00]
  Bounds range     [2e+00, 4e+00]
  RHS range        [3e+00, 8e+00]
Presolve removed 2 rows and 3 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    6.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  6.000000000e+00

Optimal Solution:
x1 = 4.0000
x2 = 0.6667
x3 = 2.0000
Objective Value = 6.0000


In [10]:
import gurobipy as gp
from gurobipy import GRB

# --- 1. 모델 생성 ---
model = gp.Model("LP_5_25")

# --- 2. 변수 정의 (각 변수의 상한/하한 포함) ---
x1 = model.addVar(lb=0, ub=3, name="x1")
x2 = model.addVar(lb=1, ub=4, name="x2")
x3 = model.addVar(lb=0, ub=8, name="x3")
x4 = model.addVar(lb=1, ub=2, name="x4")
x5 = model.addVar(lb=0, ub=4, name="x5")

# --- 3. 목적함수 (Minimize) ---
# Minimize  2x1 + 6x2 - x3 - 4x4 + x5
model.setObjective(2*x1 + 6*x2 - x3 - 4*x4 + x5, GRB.MINIMIZE)

# --- 4. 제약식 ---
model.addConstr(2*x1 + x2 + 4*x3 + x4 + x5 == 10, "c1")
model.addConstr(2*x1 + 8*x2 - 3*x3 + x4 == 7, "c2")

# --- 5. 최적화 ---
model.optimize()

# --- 6. 결과 출력 ---
if model.status == GRB.OPTIMAL:
    print("\nOptimal Solution:")
    for v in model.getVars():
        print(f"{v.varName} = {v.x:.4f}")
    print(f"Objective Value = {model.ObjVal:.4f}")
else:
    print("No optimal solution found.")

Set parameter Username
Set parameter LicenseID to value 2611964
Academic license - for non-commercial use only - expires 2026-01-20
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[arm] - Darwin 25.0.0 25A362)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2 rows, 5 columns and 9 nonzeros
Model fingerprint: 0x84749ed2
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [1e+00, 6e+00]
  Bounds range     [1e+00, 8e+00]
  RHS range        [7e+00, 1e+01]
Presolve time: 0.00s
Presolved: 2 rows, 5 columns, 9 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -3.0000000e+01   2.125000e+00   0.000000e+00      0s
       2   -2.1428571e+00   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.01 seconds (0.00 work units)
Optimal objective -2.142857143e+00

Optimal Solution:
x1 = 0.6429
x2 = 1.0000
x3 = 1.4286
x4 = 2.0000
x5 = 0.0000
Objective Va